In [ ]:
#@title Cargar datos desde jsonl a Argilla

# pip install argilla
import argilla as rg

import json
from pathlib import Path
import argilla as rg

client = rg.Argilla(
    api_url="http://localhost:6900",
    api_key="argilla.apikey",
)

DATASET_NAME = "telco_multitask_evaluation_v3"
JSONL_FILE = Path("train-dataset1") / "train.jsonl"

if not JSONL_FILE.is_file():
    raise FileNotFoundError(f"No se encontró el archivo: {JSONL_FILE.resolve()}")

# ---------- 1. Leer el JSONL una sola vez ----------
rows = []
with open(JSONL_FILE, "r", encoding="utf-8") as f:
    for line_idx, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            rows.append((line_idx, json.loads(line)))
        except json.JSONDecodeError as err:
            print(f"Línea {line_idx} omitida por JSON inválido: {err}")

# ---------- 2. Descubrir las etiquetas de "choice" ----------
choice_labels = set()
for _, d in rows:
    if d.get("primitive") == "choice":
        criteria = d.get("question", {}).get("criteria")
        if isinstance(criteria, dict):
            choice_labels.update(criteria.keys())
        if d.get("target"):
            choice_labels.add(str(d["target"]))
choice_labels = sorted(choice_labels) or ["general"]
print(f"Etiquetas detectadas para choice ({len(choice_labels)}): {choice_labels}")

# ---------- 3. Configuración del dataset ----------
# Nota: no usar "id", "user_id" ni "status" como nombre de pregunta;
# colisionan con claves reservadas del índice de búsqueda.
settings = rg.Settings(
    guidelines="Revisa la respuesta pre-cargada de cada registro, corrígela si hace falta y pulsa Submit.",
    fields=[
        rg.TextField(name="text", title="Mensaje del cliente"),
        rg.TextField(name="task_instructions", title="Instrucción de la tarea", required=False),
        rg.TextField(name="criteria_guide", title="Criterios / Rúbrica", use_markdown=True, required=False),
    ],
    questions=[
        rg.LabelQuestion(
            name="review_status",
            title="Estado del registro",
            labels=["valido", "descartar"],
            required=True,
        ),
        rg.LabelQuestion(
            name="department",
            title="Derivación de departamento",
            labels=choice_labels,
            required=False,
        ),
        rg.LabelQuestion(
            name="binary_confirmation",
            title="¿Cumple la condición planteada?",
            labels=["0", "1"],
            required=False,
        ),
        # Los valores de rating empiezan en 1: severidad 0/1/2 se muestra como 1/2/3
        rg.RatingQuestion(
            name="frustration_score",
            title="Severidad (1 = bajo, 2 = medio, 3 = alto)",
            values=[1, 2, 3],
            required=False,
        ),
    ],
    metadata=[
        rg.TermsMetadataProperty(name="primitive", title="Tipo de tarea"),
    ],
)

# ---------- 4. Crear el dataset (sin borrar nada existente) ----------
if client.datasets(DATASET_NAME) is not None:
    raise RuntimeError(f"'{DATASET_NAME}' ya existe. Usa otro nombre o bórralo desde la UI.")

dataset = rg.Dataset(name=DATASET_NAME, settings=settings, client=client)
dataset.create()
print(f"Dataset '{DATASET_NAME}' creado con éxito (id: {dataset.id}).")

# ---------- 5. Construir los registros ----------
records = []
for line_idx, d in rows:
    primitive = d.get("primitive", "")
    q = d.get("question", {})
    criteria = q.get("criteria")
    target = d.get("target")

    if isinstance(criteria, dict):
        criteria_md = "\n".join(f"- **`{k}`**: {v}" for k, v in criteria.items())
    elif isinstance(criteria, list):
        criteria_md = "\n".join(f"- **Nivel {i + 1}**: {v}" for i, v in enumerate(criteria))
    else:
        criteria_md = ""

    fields = {"text": d.get("state", "")}
    if q.get("instructions"):
        fields["task_instructions"] = q["instructions"]
    if criteria_md:
        fields["criteria_guide"] = criteria_md

    suggestions = [rg.Suggestion(question_name="review_status", value="valido")]
    if target is not None:
        if primitive == "choice":
            suggestions.append(rg.Suggestion(question_name="department", value=str(target)))
        elif primitive == "noul":
            suggestions.append(rg.Suggestion(question_name="binary_confirmation", value=str(target)))
        elif primitive == "score":
            suggestions.append(rg.Suggestion(question_name="frustration_score", value=int(target) + 1))

    records.append(
        rg.Record(
            id=str(d.get("id", f"row-{line_idx}")),
            fields=fields,
            suggestions=suggestions,
            metadata={"primitive": primitive},
        )
    )

# ---------- 6. Subida en lotes ----------
BATCH_SIZE = 100
for i in range(0, len(records), BATCH_SIZE):
    dataset.records.log(records[i:i + BATCH_SIZE])

print(f"¡Carga completada! Se subieron {len(records)} registros a '{DATASET_NAME}'.")

Etiquetas detectadas para choice (134): ['acceso_fwa_satelite', 'activacion_roaming', 'adicion_paquete_tv', 'adquisicion_b2b', 'ajuste_itinerancia', 'alta_esim', 'alta_nueva', 'alta_secundaria', 'ampliacion_cuota', 'ampliacion_velocidad', 'antena_exterior_4g', 'antivirus_red', 'averia_fisica', 'baja_administrativa', 'baja_definitiva', 'baja_linea', 'baja_parcial_lineas', 'baja_plataformas', 'baja_por_defuncion', 'baja_servicio', 'baja_servicios', 'baja_total_cuenta', 'baja_voluntaria_ordinaria', 'bajas_retenciones', 'bloqueo_datos', 'bloqueo_dns', 'bloqueo_imei', 'bloqueo_mac', 'bloqueo_sim', 'bono_datos_adicional', 'cable_cobre_adsl', 'cambio_datos_pago', 'cambio_domicilio', 'cambio_plan', 'cambio_router', 'cambio_tarifa_permanente', 'cambio_titular', 'canalizacion_obstruida', 'centralita_cloud', 'cliente_ausente', 'cobertura_movil', 'cobranzas', 'cobro_duplicado', 'cobro_equipos_no_devueltos', 'compartir_datos', 'compra_bono_extra', 'configuracion_apn', 'consulta_pin_puk', 'consumo_p

/Users/alejandro/Documents/LEARN/UGR/UGR-NLP/Argilla-docker/.venv/lib/python3.14/site-packages/argilla/client.py:356: UserWarning: Dataset with name 'telco_multitask_evaluation_v3' not found in workspace 'default'
  warnings.warn(f"Dataset with name {name!r} not found in workspace {workspace.name!r}")


Dataset 'telco_multitask_evaluation_v3' creado con éxito (id: d62891b0-dd4e-49bd-b63e-1735bca1fbbe).


DatasetRecords: The provided batch size 256 was normalized. Using value 100.

Sending records...: 100%|██████████| 1/1 [00:00<00:00,  2.29batch/s]

¡Carga completada! Se subieron 100 registros a 'telco_multitask_evaluation_v3'.


In [ ]:
#@title Exportar dataset curado

# Esto exporta el dataset a un archivo jsonl en el directorio

import json
from pathlib import Path

ds = client.datasets("telco_multitask_evaluation_v3")
rows = ds.records(with_responses=True).to_list(flatten=True)

out = Path("./train-dataset1/annotated.jsonl")

with open(out, "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")

print(f"{len(rows)} registros guardados en {out.resolve()}")

NameError: name 'client' is not defined